# FastAPI Study Notes: Employee Management API

These notes are designed for revision and hands-on practice. They explain the core FastAPI ideas first, then connect them directly to this project's `main.py` and `model.py` files.

## Learning Goals

By the end, you should be able to:

- Explain what FastAPI does and why Python type hints matter.
- Create routes for `GET`, `POST`, `PUT`, and `DELETE` operations.
- Use Pydantic models for request validation and response serialization.
- Raise meaningful HTTP errors with `HTTPException`.
- Run the app locally with Uvicorn.
- Test FastAPI routes with `TestClient`.
- Understand the Employee Management API in this repository.


## 1. What Is FastAPI?

FastAPI is a Python web framework for building APIs. It uses standard Python type hints to validate data, generate documentation, and serialize responses.

Key ideas:

| Concept | Meaning |
| --- | --- |
| ASGI app | The application object served by an ASGI server such as Uvicorn. |
| Route decorator | `@app.get(...)`, `@app.post(...)`, etc. connect a URL and HTTP method to a Python function. |
| Path parameter | A value captured from the URL, for example `/employee/{emp_id}`. |
| Query parameter | A value passed after `?`, for example `/search?q=fastapi`. |
| Request body | JSON sent by the client, commonly used with `POST` and `PUT`. |
| Response model | A Pydantic model that controls and validates the response shape. |
| OpenAPI docs | FastAPI automatically creates Swagger UI at `/docs` and ReDoc at `/redoc`. |


## 2. Project Setup

This project uses:

- `fastapi` for API routing and validation integration.
- `uvicorn[standard]` as the local development server.
- `pydantic` for data models.
- `email-validator` because the project uses Pydantic's `EmailStr` type.

Install dependencies:

```bash
pip install -r requirements.txt
```

Run the API:

```bash
uvicorn main:app --reload
```

Open the interactive docs:

```text
http://127.0.0.1:8000/docs
```


## 3. Minimal FastAPI App

Every FastAPI app starts with an `app = FastAPI()` object. Route decorators register endpoint functions on that app.


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Hello, FastAPI"}


How to read this code:

- `FastAPI()` creates the API application.
- `@app.get("/")` registers a `GET /` endpoint.
- Returning a dictionary automatically produces a JSON response.
- FastAPI inspects function arguments and type hints to validate inputs.


## 4. HTTP Methods You Use Most

| Method | Common Purpose | Example In This Project |
| --- | --- | --- |
| `GET` | Read data | `GET /employees` |
| `POST` | Create data | `POST /add_employee` |
| `PUT` | Replace or update data | `PUT /update_employee/{emp_id}` |
| `DELETE` | Remove data | `DELETE /remove_employee/{emp_id}` |

A good API uses HTTP methods to communicate intent clearly. Avoid using `GET` to create, update, or delete data.


## 5. Path Parameters And Query Parameters

Path parameters are part of the URL path and usually identify a specific resource.


In [ ]:
@app.get("/employee/{emp_id}")
def get_employee(emp_id: int):
    return {"employee_id": emp_id}


FastAPI converts `emp_id` to an integer because of `emp_id: int`. If a client sends `/employee/abc`, FastAPI returns a validation error before your function runs.

Query parameters are optional filters or modifiers after `?`.


In [ ]:
@app.get("/employees/search")
def search_employees(department: str | None = None):
    return {"department": department}


Example request:

```text
GET /employees/search?department=IT
```


## 6. Pydantic Models

Pydantic models describe the shape and validation rules for data. FastAPI uses them for request bodies, response models, and documentation.

In this project, employee data is defined in `model.py`.


In [ ]:
from pydantic import BaseModel, EmailStr, Field, StrictInt
from typing import Literal, Optional

class EmployeeBase(BaseModel):
    name: str = Field(..., min_length=3, max_length=50)
    department: Literal["HR", "IT", "Finance", "Sales"]
    email: EmailStr
    age: Optional[StrictInt] = Field(default=None, gt=0, le=120)

class EmployeeCreate(EmployeeBase):
    id: int = Field(..., gt=0)

class EmployeeUpdate(EmployeeBase):
    pass

class EmployeeResponse(EmployeeBase):
    id: int


Important validation rules in the project:

| Field | Rule |
| --- | --- |
| `id` | Required when creating an employee. Must be greater than `0`. |
| `name` | Required. Must be between `3` and `50` characters. |
| `department` | Must be one of `HR`, `IT`, `Finance`, or `Sales`. |
| `email` | Must be a valid email address. |
| `age` | Optional. If provided, must be an integer from `1` to `120`. |

`StrictInt` means Pydantic should not silently accept values like `"33"` as an integer.


## 7. Request Models vs Response Models

The project uses three employee model classes:

| Model | Used For | Why It Exists |
| --- | --- | --- |
| `EmployeeCreate` | `POST /add_employee` | Client must send an `id` when creating a record. |
| `EmployeeUpdate` | `PUT /update_employee/{emp_id}` | Client sends updated employee details; ID comes from the path. |
| `EmployeeResponse` | API responses | Response always includes the employee `id`. |

This separation keeps API contracts clear. Create, update, and response data often have similar fields, but they are not always identical.


## 8. Employee API Endpoint Map

| Method | Endpoint | Input | Success Response | Error Cases |
| --- | --- | --- | --- | --- |
| `GET` | `/` | None | Welcome message | None |
| `GET` | `/about` | None | Project description | None |
| `POST` | `/add_employee` | `EmployeeCreate` JSON | Created employee | `400` duplicate ID, `422` validation error |
| `GET` | `/employees` | None | List of employees | None |
| `GET` | `/employee/{emp_id}` | Path `emp_id` | Matching employee | `404` not found, `422` invalid path type |
| `PUT` | `/update_employee/{emp_id}` | Path `emp_id` plus `EmployeeUpdate` JSON | Updated employee | `404` not found, `422` validation error |
| `DELETE` | `/remove_employee/{emp_id}` | Path `emp_id` | Delete message | `404` not found, `422` invalid path type |


## 9. CRUD Flow In This Project

The app uses an in-memory list:

```python
employees_db: list[EmployeeResponse] = []
```

That means:

- Data exists only while the Python process is running.
- Restarting Uvicorn clears all employees.
- This is fine for learning CRUD logic.
- A real production app should use a database such as PostgreSQL, MySQL, or SQLite.


In [ ]:
from fastapi import HTTPException

employees_db = []

def find_employee(emp_id: int):
    for employee in employees_db:
        if employee.id == emp_id:
            return employee

    raise HTTPException(status_code=404, detail="Employee not found")


Study note: searching a list is simple, but it becomes slow as data grows. A database can index IDs and fetch records efficiently.


## 10. Creating An Employee

`POST /add_employee` accepts a JSON body matching `EmployeeCreate`.

Example request body:

```json
{
  "id": 101,
  "name": "Santosh Kumar",
  "department": "IT",
  "email": "santosh@company.com",
  "age": 33
}
```

Important behavior:

- The API checks whether the employee ID already exists.
- If the ID is duplicated, it returns `400 Bad Request`.
- If validation fails, FastAPI returns `422 Unprocessable Entity` automatically.


In [ ]:
from fastapi import HTTPException

@app.post("/add_employee", response_model=EmployeeResponse)
def add_employee(new_employee: EmployeeCreate):
    for employee in employees_db:
        if employee.id == new_employee.id:
            raise HTTPException(
                status_code=400,
                detail=f"Employee with EmployeeID {new_employee.id} already exists",
            )

    employee = EmployeeResponse(**new_employee.model_dump())
    employees_db.append(employee)
    return employee


## 11. Reading Employees

The project has two read endpoints:

```text
GET /employees
GET /employee/{emp_id}
```

`GET /employees` returns the full list. `GET /employee/{emp_id}` returns one employee or a `404` error.


In [ ]:
@app.get("/employees", response_model=list[EmployeeResponse])
def get_employees():
    return employees_db

@app.get("/employee/{emp_id}", response_model=EmployeeResponse)
def get_employee(emp_id: int):
    for employee in employees_db:
        if employee.id == emp_id:
            return employee

    raise HTTPException(status_code=404, detail="Employee not found")


## 12. Updating An Employee

`PUT /update_employee/{emp_id}` updates the employee whose ID appears in the path.

The request body uses `EmployeeUpdate`, so it should contain:

```json
{
  "name": "Santosh Kumar",
  "department": "Finance",
  "email": "santosh@company.com",
  "age": 34
}
```

The path controls the employee ID. The body does not include `id`.


In [ ]:
@app.put("/update_employee/{emp_id}", response_model=EmployeeResponse)
def update_employee(emp_id: int, updated_employee: EmployeeUpdate):
    for index, employee in enumerate(employees_db):
        if employee.id == emp_id:
            employees_db[index] = EmployeeResponse(
                id=emp_id,
                **updated_employee.model_dump(),
            )
            return employees_db[index]

    raise HTTPException(
        status_code=404,
        detail=f"Employee with EmployeeID {emp_id} not found",
    )


Study note: this endpoint behaves like a full update. Because `EmployeeUpdate` inherits required fields from `EmployeeBase`, the client must send all employee fields, not just the changed field.


## 13. Deleting An Employee

`DELETE /remove_employee/{emp_id}` removes one employee from the list.


In [ ]:
@app.delete("/remove_employee/{emp_id}")
def remove_employee(emp_id: int):
    for index, employee in enumerate(employees_db):
        if employee.id == emp_id:
            employees_db.pop(index)
            return {
                "message": f"Employee with EmployeeID {emp_id} successfully removed"
            }

    raise HTTPException(status_code=404, detail="Employee not found")


## 14. Status Codes To Remember

| Status Code | Meaning | Example In This Project |
| --- | --- | --- |
| `200 OK` | Request succeeded | Reading, updating, or deleting successfully. |
| `400 Bad Request` | Client sent logically invalid data | Duplicate employee ID during creation. |
| `404 Not Found` | Resource does not exist | Employee ID is not present. |
| `422 Unprocessable Entity` | Validation failed | Invalid email, invalid department, bad path parameter type. |

FastAPI automatically handles many `422` validation errors. You manually raise `HTTPException` for business rules like duplicate IDs and missing records.


## 15. Testing With TestClient

FastAPI includes a test client that lets you test routes without manually starting a server.

Create a test file such as `test_main.py` when you are ready to add automated tests.


In [ ]:
from fastapi.testclient import TestClient
from main import app, employees_db

client = TestClient(app)

def test_create_and_get_employee():
    employees_db.clear()

    payload = {
        "id": 101,
        "name": "Santosh Kumar",
        "department": "IT",
        "email": "santosh@company.com",
        "age": 33,
    }

    create_response = client.post("/add_employee", json=payload)
    assert create_response.status_code == 200
    assert create_response.json()["id"] == 101

    get_response = client.get("/employee/101")
    assert get_response.status_code == 200
    assert get_response.json()["email"] == "santosh@company.com"


Useful tests to add:

- Creating an employee succeeds with valid data.
- Creating a duplicate employee returns `400`.
- Invalid department returns `422`.
- Invalid email returns `422`.
- Fetching a missing employee returns `404`.
- Updating an existing employee changes the stored data.
- Deleting an employee removes it from `GET /employees`.


## 16. Common Mistakes And Fixes

| Mistake | Symptom | Fix |
| --- | --- | --- |
| Missing `email-validator` | Import error when using `EmailStr` | Install `email-validator`. |
| Wrong Uvicorn target | Server cannot import app | Run `uvicorn main:app --reload`. |
| Sending string for `age` | `422` validation error | Send an integer, for example `33`. |
| Sending invalid department | `422` validation error | Use `HR`, `IT`, `Finance`, or `Sales`. |
| Expecting data after restart | Employee list is empty | In-memory storage resets on restart. Use a database for persistence. |
| Sending partial update body | `422` validation error | Current `PUT` endpoint requires all update fields. |


## 17. Production Improvement Checklist

For a real employee management system, consider adding:

- Persistent database storage.
- Unique database constraint on employee ID or email.
- Authentication and authorization.
- Pagination for `GET /employees`.
- Search and filtering by department or name.
- Separate `PATCH` endpoint for partial updates.
- Structured logging.
- Automated tests with `pytest`.
- Environment-based configuration.
- Dockerfile and deployment instructions.


## 18. FastAPI Revision Cheatsheet

| Task | Pattern |
| --- | --- |
| Create app | `app = FastAPI()` |
| Register GET route | `@app.get("/path")` |
| Register POST route | `@app.post("/path")` |
| Path parameter | `def endpoint(emp_id: int)` with `/employee/{emp_id}` |
| Request body | Use a Pydantic model parameter. |
| Response model | Add `response_model=ModelName` to the decorator. |
| Raise error | `raise HTTPException(status_code=404, detail="...")` |
| Run server | `uvicorn main:app --reload` |
| Swagger UI | `/docs` |
| ReDoc | `/redoc` |
| Validation error | Usually returned as `422`. |


## 19. Practice Exercises

Try these after reading the code:

1. Add `GET /employees/department/{department}` to return employees from one department.
2. Add a check that employee emails are unique.
3. Add a `PATCH /employee/{emp_id}` endpoint for partial updates.
4. Add pagination to `GET /employees` with `limit` and `offset` query parameters.
5. Write tests for duplicate IDs, missing employees, and invalid payloads.
6. Replace the in-memory list with SQLite.

Suggested order: start with filtering, then tests, then partial updates, then database persistence.


## 20. Quick Self-Check

Answer these without looking at the code:

1. What does `response_model=EmployeeResponse` do?
2. Why does the app need `email-validator`?
3. What status code is returned for an invalid department?
4. What status code is returned for a duplicate employee ID?
5. Why does employee data disappear after restarting the server?
6. Why does `PUT /update_employee/{emp_id}` not need `id` in the request body?
7. What is the difference between a path parameter and a query parameter?
8. Where can you find FastAPI's generated Swagger UI?

If you can answer these clearly, you understand the main concepts used in this project.
